# The audit on ProsQA: Coconut LLaMA-1B (ledger §135)

**No training** of the model. Dilgren and Wiegreffe's released Coconut LLaMA-1B ProsQA checkpoint; twins
rewrite the gold path's last hop (checked by graph reachability), masking replaces thoughts by the pool
mean, native decoding throughout. The monitor layer is not evaluable on ProsQA twins and the card says so.
No dataset to attach; the 500-row test file is fetched from the authors' repository and hash-checked.
Publish the output as a dataset when finished (Output tab -> New Dataset).

## Setup

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "c71f3cc562d6c935c9172f88a3ead39118049c51"
REPO_DIR = "/kaggle/working/latent-reasoning"
FAMILY = "coconut"
PRECISION = "float16"
DATA_URL = "https://raw.githubusercontent.com/connordilgren/are-lrms-easily-interpretable/main/data/prosqa_test.json"
DATA_SHA256 = "82ddc0afe6ee30eb31bb16fa208e293e55278a20fa987098d48519cae7e12680"
OUTPUT_ROOT = "/kaggle/working/prosqa_coconut"
RUN_SMOKE = True

import hashlib, json, os, pathlib, subprocess, sys, time, urllib.request
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
assert len(RUN_COMMIT) == 40 and RUN_COMMIT != "0" * 40
if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", RUN_COMMIT], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("code commit", subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())
print("gpu", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
DATA = "/kaggle/working/prosqa_test.json"
urllib.request.urlretrieve(DATA_URL, DATA)
got = hashlib.sha256(open(DATA, "rb").read()).hexdigest()
assert got == DATA_SHA256, f"ProsQA test file changed upstream: {got}"
print("prosqa_test.json", got[:12], len(json.load(open(DATA))), "rows")

### Environment and focused tests

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "tpot"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.52.4", "datasets==3.6.0", "peft==0.15.2", "accelerate==1.7.0",
                "huggingface-hub>=0.34,<1.0", "safetensors==0.5.3", "pytest>=8,<10"], check=True)
probe = subprocess.run([sys.executable, "-c",
    "from peft.import_utils import is_torchao_available\ntry:\n print(is_torchao_available())\nexcept ImportError:\n print('incompatible')"],
    capture_output=True, text=True)
if "incompatible" in probe.stdout:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", *["tests/test_prosqa.py", "tests/test_coconut.py", "tests/test_prosqa_audit_runner.py", "tests/test_official_codi_family.py"]], check=True)

def run_with_retry(command, attempts=3, wait=60):
    for attempt in range(1, attempts + 1):
        result = subprocess.run(command)
        if result.returncode == 0:
            return
        if attempt == attempts:
            raise subprocess.CalledProcessError(result.returncode, command)
        print(f"runner exited with {result.returncode}; retrying in {wait}s ({attempt + 1}/{attempts})")
        time.sleep(wait)

## The audit (smoke first, then all 500 rows)

In [ ]:
def audit(out, smoke):
    cmd = [sys.executable, "-u", "scripts/run_prosqa_audit.py", "--family", FAMILY, "--data", DATA, "--output-dir", out,
           "--batch-size", "16", "--precision", PRECISION, "--device", "cuda"]
    if smoke:
        cmd.append("--smoke")
    run_with_retry(cmd)
    return json.loads(open(pathlib.Path(out) / "summary.json").read())

if RUN_SMOKE:
    audit(OUTPUT_ROOT + "_smoke", True)
summary = audit(OUTPUT_ROOT, False)
from IPython.display import Markdown, display
display(Markdown(open(f"{OUTPUT_ROOT}/card.md").read()))
print(json.dumps(summary["checks"], indent=2))
print("Reminder: publish this notebook's output as a dataset (Output tab -> New Dataset).")

## Keep the output under Kaggle's 500-file cap

In [ ]:
import shutil
shutil.rmtree(REPO_DIR, ignore_errors=True)   # the clone (with .git) is not an output; without this the run's own files can be dropped from the saved output
print("removed", REPO_DIR, "; remaining output files:", sum(len(f) for _, _, f in os.walk("/kaggle/working")))